# Migration: SRC_DEPARTMENTS to TRG_DEPARTMENTS
**Source File:** Dept_pkg_conversion.sql
**Target Platform:** Databricks Spark SQL (Delta Lake)

In [ ]:
dbutils.widgets.text("ODI_SESS_NO", "")
dbutils.widgets.text("ETL_PROC_WID", "")
dbutils.widgets.text("DATASOURCE_NUM_ID", "")

### ETL Parameters

In [ ]:
%sql
-- SCEN_TASK_NO {1}: Get last run date
CREATE OR REPLACE TEMPORARY VIEW v_last_run_params AS
SELECT 
    COALESCE(
        (SELECT date_format(last_run_dt, 'dd-MM-yy') 
         FROM workspace.odi_trg.log_table1 
         WHERE pkg_name = 'Dept_pkg'), 
        date_format(current_timestamp(), 'dd-MM-yy')
    ) AS last_run_dt;

In [ ]:
%sql
-- SCEN_TASK_NO {2}: Update log table
UPDATE workspace.odi_trg.log_table1 
SET last_run_dt = current_timestamp() 
WHERE pkg_name = 'Dept_pkg';

### Staging Table Handling

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Drop staging table
DROP TABLE IF EXISTS workspace.odi_trg.c_0src_departments;

In [ ]:
%sql
-- SCEN_TASK_NO {40}: Create staging table
CREATE TABLE workspace.odi_trg.c_0src_departments
(
	DEPARTMENT_ID	BIGINT,
	DEPARTMENT_NAME	STRING,
	MANAGER_ID	BIGINT,
	LOCATION_ID	BIGINT,
	LAST_UPD_DT	TIMESTAMP
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {50}: Insert into staging
INSERT INTO workspace.odi_trg.c_0src_departments
(
	DEPARTMENT_ID,
	DEPARTMENT_NAME,
	MANAGER_ID,
	LOCATION_ID,
	LAST_UPD_DT
)
SELECT	
	SRC_DEPARTMENTS_1.DEPARTMENT_ID,
	SRC_DEPARTMENTS_1.DEPARTMENT_NAME,
	SRC_DEPARTMENTS_1.MANAGER_ID,
	SRC_DEPARTMENTS_1.LOCATION_ID,
	SRC_DEPARTMENTS_1.LAST_UPD_DT
FROM workspace.odi_src.src_departments AS SRC_DEPARTMENTS_1
WHERE (1=1);

In [ ]:
%sql
-- SCEN_TASK_NO {60}: Table Optimization
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_trg.c_0src_departments;

### Target Load

In [ ]:
%sql
-- SCEN_TASK_NO {80}: Load target table
INSERT INTO workspace.odi_trg.trg_departments
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
) 
SELECT 
    SRC_DEPARTMENTS_A.DEPARTMENT_ID,
    SRC_DEPARTMENTS_A.DEPARTMENT_NAME,
    SRC_DEPARTMENTS_A.MANAGER_ID,
    SRC_DEPARTMENTS_A.LOCATION_ID,
    SRC_DEPARTMENTS_A.LAST_UPD_DT  
FROM 
    workspace.odi_trg.c_0src_departments AS SRC_DEPARTMENTS_A;

### Cleanup

In [ ]:
%sql
-- SCEN_TASK_NO {110}: Cleanup staging table
DROP TABLE IF EXISTS workspace.odi_trg.c_0src_departments;